# Análise Exploratória dos Dados (AED)

Este é o primeiro de dois notebooks do projeto. Ele faz a **Análise Exploratória de Dados (Exploratory Data Analysis, EDA)** do conjunto usado para modelar o valor mediano dos imóveis (`MEDV`).

Esta etapa estuda a **estrutura, qualidade e distribuição dos dados** e as relações entre as variáveis, antes dos modelos estatísticos ou de aprendizado de máquina. Ela também procura problemas possíveis: valores faltantes, duplicatas, assimetria, observações discrepantes, relações não lineares e multicolinearidade.

A análise tem quatro partes:

## 1. Qualidade e estrutura dos dados

Primeiro, a análise faz uma inspeção geral do conjunto de dados. Ela examina:

- As dimensões do dataset
- Os tipos das variáveis
- Os valores faltantes
- As observações duplicadas
- As estatísticas descritivas

A análise também identifica a **natureza de cada variável**. Algumas variáveis têm características que a modelagem precisa considerar:

- `CHAS` é uma variável binária
- `RAD` é um índice discreto com poucos valores distintos
- `ZN` tem muitos valores iguais a zero

A análise também estuda a variável resposta `MEDV`. Essa variável tem **censura no valor 50**: cerca de 16 observações são exatamente iguais a `50.0`. Essa concentração no limite superior pode influenciar a análise dos resíduos. Por isso, a modelagem precisa considerá-la.

## 2. Análise univariada

Depois, a análise estuda a distribuição de cada variável, uma de cada vez. Ela usa:

- Histogramas e estimativas de densidade
- Medidas de assimetria
- Medidas de curtose

A variável `MEDV` recebe atenção especial. Se a assimetria à direita for grande, a análise avalia a transformação


log(MEDV)


como alternativa. O objetivo é uma distribuição mais adequada e, possivelmente, resíduos mais próximos da normalidade e da homocedasticidade.

Alguns preditores, como `CRIM`, `LSTAT`, `DIS` e `NOX`, também podem ter assimetria forte. Para essas variáveis, a análise avalia transformações, como o **logaritmo**, para obter distribuições mais adequadas à modelagem.

## 3. Relação de cada preditor com `MEDV`

Em seguida, a análise estuda a relação entre cada preditor e a variável resposta `MEDV`.

Ela usa gráficos de dispersão de `MEDV` em função de cada preditor, com uma curva suavizada **LOWESS**. A curva mostra se cada relação é aproximadamente linear ou se tem um padrão não linear.

A análise também usa:

- Boxplots de `MEDV` para os valores de `CHAS`
- Boxplots de `MEDV` para possíveis faixas de `RAD`
- Padrões de agrupamento ou concentração de observações

Por exemplo, a análise espera uma relação aproximadamente linear e positiva entre `RM` e `MEDV`, e talvez uma relação mais curva em `LSTAT`. Esses padrões podem indicar a necessidade de transformações ou de termos não lineares, como termos quadráticos.

As variáveis `TAX` e `RAD` também recebem atenção especial. Nelas, podem existir grupos de observações com valores idênticos. Esses grupos podem influenciar a leitura das relações e dos modelos ajustados.

## 4. Multicolinearidade

Por fim, a análise estuda a dependência entre os preditores, para identificar problemas de **multicolinearidade**. Ela usa:

- A matriz de correlação de **Pearson**
- Heatmaps das correlações
- O **Variance Inflation Factor (VIF)**
- O número de condição da matriz de design

A análise calcula o VIF de cada variável, com atenção especial aos valores altos. Valores acima de **5 a 10** podem indicar multicolinearidade relevante.

Os resultados desta etapa orientam a seleção e a transformação das variáveis no modelo. Entre as opções estão **remover ou combinar variáveis** e usar métodos de regularização, como **Ridge** e **Lasso**.

---

> **Objetivo final da EDA:** conhecer as características dos dados e identificar os padrões e os problemas dos dados. É necessário considerar esses pontos antes de construir e avaliar os modelos de regressão.

> **Próximo notebook:** [`02-MRLM.ipynb`](02-MRLM.ipynb) ajusta o modelo de regressão linear múltipla (Partes 5 a 12). Ele lê o arquivo `data/housing_processed.csv`, que este notebook salva no final. Rode este notebook primeiro.

In [2]:
import os
import pandas as pd
import numpy as np
from statsmodels.tools.tools import add_constant
from scipy.stats import mannwhitneyu, kruskal
import plotly.graph_objects as go

# Funções de gráfico compartilhadas com o notebook 02-MRLM (notebooks/graficos.py)
from graficos import (
    COR_PONTO, configurar, mostrar_e_salvar, gerar_subplot_hist, gerar_subplot_regplot,
    grafico_categoria, heatmap_correlacao,
)

In [3]:
output_fig_dir = '../results/figures'
os.makedirs(output_fig_dir, exist_ok=True)

In [4]:
df = pd.read_csv("../data/housing.csv")

# CHAS é uma variável indicadora (0/1): tratada como categórica, não contínua
df["CHAS"] = df["CHAS"].astype("category")

# RAD é um índice com 9 níveis (1 a 8 e 24): tratado como categórico ordinal, não contínuo
df["RAD"] = df["RAD"].astype(pd.CategoricalDtype(ordered=True))

In [5]:
# Gráficos interativos (Plotly), definidos em graficos.py. Passe o mouse sobre um ponto para ver
# a linha do arquivo, a cidade e os valores da observação. Clique na legenda para esconder ou mostrar uma série.
configurar(df, output_fig_dir)


def calcular_vif(df: pd.DataFrame, colunas: list[str]):
    # VIF generalizado (GVIF, Fox & Monette, 1992): uma variável categórica com k níveis
    # entra como k-1 indicadoras, e o GVIF mede a colinearidade do bloco inteiro.
    # Para variáveis numéricas e binárias (GL = 1), o GVIF é igual ao VIF.
    X = df[colunas].dropna().copy()

    blocos = {}
    partes = []
    for coluna in colunas:
        if isinstance(X[coluna].dtype, pd.CategoricalDtype):
            parte = pd.get_dummies(X[coluna], prefix=coluna, drop_first=True, dtype=float)
        else:
            parte = X[[coluna]].astype(float)
        blocos[coluna] = list(parte.columns)
        partes.append(parte)

    X_design = pd.concat(partes, axis=1)
    R = X_design.corr()
    det_R = np.linalg.det(R)

    linhas = []
    for coluna, colunas_bloco in blocos.items():
        outras = [c for c in X_design.columns if c not in colunas_bloco]
        gvif = (
            np.linalg.det(R.loc[colunas_bloco, colunas_bloco])
            * np.linalg.det(R.loc[outras, outras])
            / det_R
        )
        gl = len(colunas_bloco)
        linhas.append({
            "Variável": coluna,
            "GL": gl,
            "GVIF": gvif,
            # Comparável a sqrt(VIF): 2,24 equivale a VIF 5 e 3,16 a VIF 10
            "GVIF^(1/(2·GL))": gvif ** (1 / (2 * gl)),
        })

    return (
        pd.DataFrame(linhas)
        .sort_values(by="GVIF^(1/(2·GL))", ascending=False)
        .reset_index(drop=True)
    )

def calcular_numero_condicao(df: pd.DataFrame, colunas: list[str]):
    X = df[colunas].dropna().copy()
    X = pd.get_dummies(X, drop_first=True, dtype=float)
    # Padroniza para que o número de condição não dependa das unidades
    X = (X - X.mean()) / X.std()
    X_com_const = add_constant(X)
    return np.linalg.cond(X_com_const.values)

## **Parte 1: Qualidade e estrutura dos dados**

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 511 entries, 0 to 510
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   CRIM     511 non-null    float64 
 1   ZN       511 non-null    float64 
 2   INDUS    511 non-null    float64 
 3   CHAS     511 non-null    category
 4   NOX      511 non-null    float64 
 5   RM       506 non-null    float64 
 6   AGE      511 non-null    float64 
 7   DIS      511 non-null    float64 
 8   RAD      511 non-null    category
 9   TAX      511 non-null    int64   
 10  PTRATIO  511 non-null    float64 
 11  B        511 non-null    float64 
 12  LSTAT    511 non-null    float64 
 13  MEDV     511 non-null    float64 
dtypes: category(2), float64(11), int64(1)
memory usage: 49.1 KB


In [7]:
df.describe()

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,TAX,PTRATIO,B,LSTAT,MEDV
count,511.000000,511.000000,511.000000,511.000000,506.000000,511.000000,511.000000,511.000000,511.000000,511.000000,511.000000,511.000000
mean,3.584139,11.252446,11.151096,0.554757,6.287589,68.616243,3.783876,407.440313,18.500000,356.600900,12.879550,22.682192
std,8.564433,23.234838,6.828175,0.115310,0.703802,28.099130,2.098631,167.903532,2.200348,90.882679,7.797416,9.484262
min,0.006320,0.000000,0.460000,0.385000,3.561000,2.900000,1.129600,187.000000,12.600000,0.320000,1.730000,5.000000
25%,0.082325,0.000000,5.190000,0.449000,5.885500,45.050000,2.100350,279.500000,17.400000,374.710000,7.065000,17.050000
50%,0.261690,0.000000,9.690000,0.538000,6.209000,77.300000,3.152300,330.000000,19.100000,391.340000,11.450000,21.200000
75%,3.621175,12.500000,18.100000,0.624000,6.629750,94.050000,5.118000,666.000000,20.200000,396.210000,17.105000,25.000000
max,88.976200,100.000000,27.740000,0.871000,8.780000,100.000000,12.126500,711.000000,23.000000,396.900000,76.000000,67.000000


In [8]:
pd.DataFrame({
    'Frequência': df['CHAS'].value_counts().sort_index(),
    'Proporção': df['CHAS'].value_counts(normalize=True).sort_index().round(3)
})

,Frequência,Proporção
CHAS,,
0,476,0.932
1,35,0.068


In [9]:
pd.DataFrame({
    'Frequência': df['RAD'].value_counts().sort_index(),
    'Proporção': df['RAD'].value_counts(normalize=True).sort_index().round(3)
})

,Frequência,Proporção
RAD,,
1,20,0.039
2,24,0.047
3,43,0.084
4,110,0.215
5,115,0.225
6,26,0.051
7,17,0.033
8,24,0.047
24,132,0.258


In [10]:
for i in df.columns:
    print(f"Valores únicos de {i}: {df[i].nunique()}")

Valores únicos de CRIM: 509
Valores únicos de ZN: 26
Valores únicos de INDUS: 79
Valores únicos de CHAS: 2
Valores únicos de NOX: 82
Valores únicos de RM: 444
Valores únicos de AGE: 357
Valores únicos de DIS: 416
Valores únicos de RAD: 9
Valores únicos de TAX: 67
Valores únicos de PTRATIO: 47
Valores únicos de B: 360
Valores únicos de LSTAT: 460
Valores únicos de MEDV: 231


In [11]:
print((df["MEDV"] == 50.0).sum())
df["MEDV"].value_counts().sort_index(ascending=False).head(10)

16


MEDV
67.0     1
54.0     1
50.0    16
48.8     1
48.5     1
48.3     1
46.7     1
46.0     1
45.4     1
44.8     1
Name: count, dtype: int64

In [12]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")

Linhas duplicadas: 0


Observações com `RM` ausente:

In [13]:
df[df["RM"].isna()]

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
10,0.22489,12.5,7.87,0,0.524,NaN,94.3,6.3467,5,311,15.2,392.52,20.45,15.0
35,0.06417,0.0,5.96,0,0.499,NaN,68.2,3.3603,5,279,19.2,396.90,9.68,18.9
63,0.12650,25.0,5.13,0,0.453,NaN,43.4,7.9809,8,284,19.7,395.58,9.50,25.0
96,0.11504,0.0,2.89,0,0.445,NaN,69.6,3.4952,2,276,18.0,391.83,11.34,21.4
135,0.55778,0.0,21.89,0,0.624,NaN,98.2,2.1107,4,437,21.2,394.67,16.96,18.1


Consistência com o domínio das variáveis. No Boston Housing original, o valor máximo de `MEDV` é 50, e `LSTAT` (percentual) não passa de ~38. As observações fora desses limites são suspeitas:

In [14]:
suspeitas = df[(df["MEDV"] > 50) | (df["LSTAT"] > 40)]
suspeitas

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
508,0.44433,0.0,12.5,0,0.561,6.123,98.0,2.987,3,320,23.0,343.0,21.0,54.0
509,0.77763,0.0,12.7,0,0.561,6.222,34.0,2.543,3,329,23.0,343.0,76.0,67.0
510,0.65432,0.0,12.8,0,0.561,6.760,67.0,2.987,3,345,23.0,321.0,45.0,24.0


O dataset original tem 506 observações (índices 0 a 505). A tabela a seguir mostra as 5 linhas extras deste arquivo (índices 506 a 510):

In [15]:
df.loc[506:]

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
506,0.98765,0.0,12.5,0,0.561,6.980,89.0,2.098,3,320,23.0,396.0,12.0,12.0
507,0.23456,0.0,12.5,0,0.561,6.980,76.0,2.654,3,320,23.0,343.0,25.0,32.0
508,0.44433,0.0,12.5,0,0.561,6.123,98.0,2.987,3,320,23.0,343.0,21.0,54.0
509,0.77763,0.0,12.7,0,0.561,6.222,34.0,2.543,3,329,23.0,343.0,76.0,67.0
510,0.65432,0.0,12.8,0,0.561,6.760,67.0,2.987,3,345,23.0,321.0,45.0,24.0


Número de valores fora das cercas de Tukey (Q1 − 1,5·IQR, Q3 + 1,5·IQR) em cada variável numérica:

In [16]:
numericas = df.select_dtypes(include="number").columns
q1 = df[numericas].quantile(0.25)
q3 = df[numericas].quantile(0.75)
iqr = q3 - q1
fora = (df[numericas] < q1 - 1.5 * iqr) | (df[numericas] > q3 + 1.5 * iqr)

pd.DataFrame({
    "Outliers (IQR)": fora.sum(),
    "Proporção": fora.mean().round(3)
}).sort_values("Outliers (IQR)", ascending=False)

,Outliers (IQR),Proporção
B,76,0.149
ZN,68,0.133
CRIM,67,0.131
MEDV,42,0.082
RM,30,0.059
PTRATIO,15,0.029
LSTAT,8,0.016
DIS,5,0.010
NOX,0,0.000
INDUS,0,0.000


In [17]:

fig = go.Figure(go.Histogram(
    x=df["MEDV"], marker=dict(color=COR_PONTO, line=dict(width=0.5, color="#fcfcfb")),
    xbins=dict(start=df["MEDV"].min(), end=df["MEDV"].max(), size=(df["MEDV"].max() - df["MEDV"].min()) / 100),
    hovertemplate="MEDV: %{x}<br>Frequência: %{y}<extra></extra>",
))
fig.update_xaxes(title_text="MEDV")
fig.update_yaxes(title_text="Frequência")
mostrar_e_salvar(fig, altura=400)

### Conclusão - Qualidade e estrutura dos dados

O conjunto de dados tem **511 observações e 14 variáveis**. São 11 variáveis do tipo `float64`, 1 do tipo `int64` (`TAX`) e 2 convertidas para o tipo `category` (`CHAS` e `RAD`). Em geral, a estrutura dos dados é adequada para a análise, e os valores ausentes são poucos.

A variável `RM` tem **5 valores ausentes**. As outras variáveis têm 511 observações não nulas. É necessário tratar esses valores antes da modelagem. A estratégia (por exemplo, imputação ou remoção das observações) ainda precisa de avaliação.

O número de valores únicos mostra que as variáveis têm naturezas diferentes. `CHAS` tem só **2 valores distintos**: é uma variável binária. Por isso, a análise a trata como **categórica**. A média e o desvio padrão de `CHAS` não têm leitura de variável contínua. A descrição correta é a frequência de cada categoria: 476 observações (93,2%) com `CHAS = 0` e só 35 (6,8%) com `CHAS = 1`.

`RAD` tem só **9 valores distintos** (1 a 8 e 24). É um índice de acessibilidade, não uma medida, e os seus valores funcionam como rótulos ordenados. Por exemplo, o salto de 8 para 24 não indica uma acessibilidade três vezes maior. Por isso, a análise trata `RAD` como **categórica ordinal**. As categorias não têm o mesmo tamanho: `RAD = 24` tem 132 observações (25,8%), `RAD = 5` tem 115 (22,5%) e `RAD = 4` tem 110 (21,5%). `RAD = 7` tem só 17 (3,3%).

`ZN` tem só **26 valores distintos**, com muitas observações concentradas em poucos valores. As outras variáveis têm mais valores distintos, e a análise as trata, em geral, como contínuas. Por exemplo, `CRIM` tem 509 valores únicos, `LSTAT` tem 460 e `RM` tem 444, entre as 506 observações disponíveis.

A variável resposta `MEDV` tem uma característica importante: **16 observações são exatamente iguais a 50,0**, e os valores próximos têm muito menos observações. Essa concentração no limite superior indica **censura no valor 50**. A censura pode afetar a distribuição da variável resposta e, como resultado, os resíduos e os pressupostos dos modelos de regressão. Mas dois valores estão **acima** do limite de censura (`MEDV` = 54 e 67). Esses valores não são compatíveis com a censura em 50, e por isso a análise examinou a consistência com o domínio das variáveis.

Não há observações duplicadas. Mas o exame de consistência mostrou que as **5 últimas linhas do arquivo (índices 506 a 510)** são suspeitas. O Boston Housing original tem exatamente 506 observações. Essas linhas extras têm estas características:

- Os valores de `CRIM` têm um padrão sequencial de dígitos (0,98765, 0,23456, 0,44433, 0,77763 e 0,65432)
- `AGE`, `B`, `LSTAT` e `MEDV` são todos inteiros, ao contrário do restante do conjunto
- `MEDV` = 54 e 67, acima do limite de censura em 50
- `LSTAT` = 76 e 45, muito acima do máximo de ~38 do conjunto original (um percentual de 76 não é plausível para essa variável)

A análise **manteve** essas observações nesta etapa, mas as sinalizou. Elas afetam a assimetria, a curtose, as correlações, o VIF e o teste de Mann-Whitney. Antes do ajuste do modelo, é necessário avaliar a remoção delas.

A análise também manteve as 5 observações com `RM` ausente (índices 10, 35, 63, 96 e 135). Até o tratamento dessas linhas, o cálculo do VIF as descarta (`dropna`), e as matrizes de correlação usam exclusão par a par. Como resultado, os dois cálculos não usam exatamente as mesmas linhas.

As cercas de Tukey (1,5·IQR) mostram mais valores extremos em `B` (76 observações), `ZN` (68), `CRIM` (67) e `MEDV` (42). Em `ZN` e `B`, a causa principal é a forte concentração da distribuição (zeros em `ZN` e valores próximos de 396,9 em `B`). Esses valores não são necessariamente erros nos dados. Em `CRIM` e `MEDV`, a causa é a assimetria à direita.

Em resumo, a estrutura geral do conjunto é adequada. Mas as próximas etapas precisam tratar quatro pontos:

- **Os valores ausentes em `RM`**
- **As 5 observações suspeitas (índices 506 a 510)**
- **A natureza discreta ou concentrada de algumas variáveis**
- **A censura de `MEDV` em 50**

A análise univariada e o estudo das relações entre as variáveis consideram esses pontos.

## **Parte 2: Análise univariada**

In [18]:
variaveis_hist = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX',
       'PTRATIO', 'B', 'LSTAT', 'MEDV']

gerar_subplot_hist(
    variaveis=variaveis_hist,
    size=(15, 10),
    name='hist_variaveis_sem_tranformacoes'
)

In [19]:
df["logCRIM"] = np.log1p(df["CRIM"])
df["logZN"] = np.log1p(df["ZN"])
df["logDIS"] = np.log(df["DIS"])
df["logLSTAT"] = np.log(df["LSTAT"])
df["logMEDV"] = np.log(df["MEDV"])

In [20]:
novas_variaveis_hist = ["logCRIM", "logZN", "logDIS", "logLSTAT", "logMEDV"]

gerar_subplot_hist(
    variaveis=novas_variaveis_hist,
    size=(15, 8),
    name='hist_variaveis_transformadas'
)

### Conclusão - Análise Univariada

Os histogramas, as distribuições e as medidas de assimetria mostram que algumas variáveis são muito assimétricas. Essas variáveis são candidatas a transformações antes da modelagem.

A variável `CRIM` tem **assimetria positiva forte**, com valor de 5,25. Com a transformação logarítmica, a assimetria de `logCRIM` cai para 1,28, uma melhora grande. `B` tem assimetria negativa forte, de -2,90: as observações se concentram numa região, com uma cauda longa à esquerda.

A variável resposta `MEDV` também tem assimetria positiva, com valor de 1,23. Com a transformação logarítmica, a assimetria de `logMEDV` passa para -0,28, uma distribuição muito mais simétrica. Esse resultado reforça a possibilidade de usar **`log(MEDV)` como variável resposta** nos modelos. Mas a escolha final também depende dos diagnósticos dos resíduos e da censura no valor 50.

Em `LSTAT`, a assimetria passa de 1,81 para -0,25 com a transformação logarítmica. Assim, o logaritmo também é uma alternativa promissora para esse preditor.

`ZN` continua bimodal depois da transformação logarítmica. A causa é a grande concentração de valores iguais a zero, uma característica estrutural que a transformação não resolve. Por isso, o tratamento de `ZN` precisa considerar a distribuição dos valores e, talvez, separar as observações iguais a zero das observações com valores positivos.

Em resumo, **`CRIM`, `MEDV` e `LSTAT` são candidatos relevantes à transformação logarítmica**. `B` precisa de um estudo mais detalhado, por causa da assimetria negativa. `ZN` precisa de uma abordagem que considere a concentração de zeros e a distribuição bimodal. É necessário avaliar essas transformações também nas relações entre as variáveis e no desempenho dos modelos. Uma distribuição mais simétrica, sozinha, não garante os pressupostos da regressão.

## **Parte 3: Relação de cada preditor com MEDV**

In [21]:
variaveis_regplot = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX',
       'PTRATIO', 'B', 'LSTAT', "logCRIM", "logZN", "logDIS", "logLSTAT"]

gerar_subplot_regplot(
    variaveis=variaveis_regplot,
    y='MEDV',
    size=(15, 10),
    name='MEDV_vs_variaveis'
)

/home/ribamarneto/Documentos/Trabalho Ronald/.venv/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:237: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(


In [22]:
gerar_subplot_regplot(
    variaveis=variaveis_regplot,
    y='logMEDV',
    size=(15, 10),
    name='logMEDV_vs_variaveis'
)

In [23]:
grafico_categoria('CHAS', 'MEDV_vs_CHAS')

In [24]:
resumo_chas = df.groupby('CHAS', observed=True)['MEDV'].agg(['count', 'median', 'mean', 'std'])
print(resumo_chas)

medv_chas_0 = df.loc[df['CHAS'] == 0, 'MEDV']
medv_chas_1 = df.loc[df['CHAS'] == 1, 'MEDV']

estatistica, p_valor = mannwhitneyu(medv_chas_1, medv_chas_0, alternative='two-sided')
print(f"\nMann-Whitney U: U = {estatistica:.1f} | p-valor = {p_valor:.4f}")

      count  median       mean        std
CHAS                                     
0       476   20.95  22.258824   9.163078
1        35   23.30  28.440000  11.816643

Mann-Whitney U: U = 10942.5 | p-valor = 0.0019


In [25]:
grafico_categoria('RAD', 'MEDV_vs_RAD')

In [26]:
resumo_rad = df.groupby('RAD', observed=True)['MEDV'].agg(['count', 'median', 'mean', 'std'])
print(resumo_rad)

grupos_rad = [grupo['MEDV'] for _, grupo in df.groupby('RAD', observed=True)]

estatistica, p_valor = kruskal(*grupos_rad)
print(f"\nKruskal-Wallis: H = {estatistica:.2f} | p-valor = {p_valor:.4g}")

     count  median       mean        std
RAD                                     
1       20   22.20  24.365000   8.024454
2       24   23.85  26.833333   7.874376
3       43   26.60  29.076744  10.910302
4      110   20.45  21.387273   6.957883
5      115   23.00  25.706957   9.328401
6       26   21.20  20.976923   2.312801
7       17   26.20  27.105882   6.493215
8       24   28.25  30.358333   9.727724
24     132   14.40  16.403788   8.539745

Kruskal-Wallis: H = 157.18 | p-valor = 6.214e-30


### Conclusão - Relação dos Preditores com `MEDV`

Os gráficos de dispersão, com as curvas LOWESS, mostram que as relações entre os preditores e a variável resposta `MEDV` têm graus diferentes de linearidade. Esse resultado indica quais variáveis podem precisar de transformações ou de termos não lineares nos modelos de regressão.

As variáveis `CRIM` e `LSTAT` têm relações claramente não lineares com `MEDV`. Nos dois casos, as transformações logarítmicas (`logCRIM` e `logLSTAT`) deixam as relações mais próximas de uma reta. Assim, essas transformações são alternativas promissoras para a modelagem.

A variável `DIS` tem uma relação em forma de joelho: um crescimento inicial mais forte e depois uma tendência à estabilização. A transformação `logDIS` suaviza esse comportamento, mas ainda há curvatura. Por isso, vale avaliar um termo quadrático como alternativa à transformação logarítmica.

Em `ZN`, a transformação `logZN` mostra um padrão mais monotônico. Mas a grande concentração de valores iguais a zero, vista na análise univariada, dificulta o tratamento de `ZN` como variável contínua comum. Uma alternativa a investigar é uma variável indicadora que separe as observações com `ZN = 0` das observações com `ZN > 0`. Assim, é possível avaliar em separado o efeito dos valores positivos.

`RM` tem uma relação aproximadamente linear e positiva com `MEDV` na escala original. A princípio, não há motivo claro para transformá-la. `RM` pode ficar na forma original, e os diagnósticos do modelo avaliam essa escolha depois.

A variável `B` exige cuidado. A distribuição de `B` é muito concentrada perto de 400, com uma cauda longa em direção a zero. A associação de `B` com `MEDV` deve ser lida com essa distribuição em mente, com possíveis agrupamentos ou particularidades dos dados, e sem supor uma relação causal. Por isso, a modelagem avalia a contribuição de `B` com atenção.

#### Análise de `CHAS` e `RAD` como fatores de agrupamento

`CHAS` é binária. Por isso, a análise compara os dois grupos, em vez de usar um gráfico de dispersão ou uma correlação. O boxplot e o resumo por grupo mostram uma mediana maior para `CHAS = 1` (23,30, média 28,44) do que para `CHAS = 0` (20,95, média 22,26).

O teste de Mann-Whitney é adequado por causa da assimetria e da censura de `MEDV`. Ele indica diferença significativa entre as distribuições (U = 10942,5, p-valor = 0,0019). Esse resultado sugere uma associação entre a proximidade do rio Charles e o valor mediano dos imóveis. Mas o grupo `CHAS = 1` é pequeno (35 observações), e a leitura exige cuidado. Assim, `CHAS` continua candidata a preditora, e os modelos ajustados avaliam a sua contribuição.

Pelo mesmo motivo, a análise compara os níveis de `RAD`, em vez de usar um gráfico de dispersão. A mediana de `MEDV` não segue a ordem do índice. Nos níveis de 1 a 8, ela varia entre 20,45 (`RAD = 4`) e 28,25 (`RAD = 8`). Em `RAD = 24`, o nível com mais observações (132), ela cai para **14,40**.

O teste de Kruskal-Wallis é a extensão do Mann-Whitney para mais de dois grupos. Ele indica diferença significativa entre os níveis (H = 157,18, p-valor < 0,001). A relação não é monotônica, e por isso uma única inclinação para `RAD` numérica não representa esse padrão. Além disso, a forte associação de `RAD` com `TAX` também importa. A análise de multicolinearidade estuda essa associação.

Assim, a análise representa `RAD` como **fator categórico** (8 variáveis indicadoras, com `RAD = 1` como referência). Alguns níveis têm poucas observações (17 em `RAD = 7`, 20 em `RAD = 1`). Por isso, a modelagem pode avaliar o agrupamento dos níveis, por exemplo `RAD = 24` contra os outros. Essa decisão precisa de apoio nos dados e de uma avaliação da contribuição para o modelo.

Em resumo, **`logCRIM` e `logLSTAT` são alternativas promissoras para aproximar relações não lineares de relações lineares**. `DIS` pode precisar de uma abordagem adicional para representar a sua curvatura. `RM` pode ficar na escala original no início. `ZN`, `B`, `CHAS` e `RAD` precisam de tratamentos específicos, de acordo com as suas características e os padrões observados.

Essas conclusões orientam as formas funcionais que a modelagem testa. A decisão final considera o formato dos gráficos e também a significância dos termos, o ajuste dos modelos e os diagnósticos dos resíduos.

## **Parte 4: Multicolinearidade**

> `CHAS` não entra na matriz de correlação de Pearson, porque é uma variável categórica (binária). A Parte 3 avaliou a associação de `CHAS` com `MEDV` pela comparação entre grupos. O VIF avalia a relação de `CHAS` com os outros preditores, com `CHAS` como variável indicadora.

In [27]:
corr = df.drop(columns='CHAS').corr(numeric_only=True)
heatmap_correlacao(corr, 'Matriz de Correlação', 'heatmap_de_correlacao')

Várias variáveis são assimétricas ou discretas (`CRIM`, `ZN`, `RAD`, `TAX`). Por isso, a correlação de **Spearman** (baseada em postos) complementa a de Pearson. A correlação de Spearman capta relações monotônicas não lineares e é menos sensível a valores extremos. `RAD` é categórica e não entra na matriz de Pearson. Mas `RAD` é ordinal, e por isso entra na de Spearman, que depende só da ordem dos níveis.

In [28]:
# RAD é ordinal: seus códigos preservam a ordem dos níveis, o que basta para Spearman
corr_spearman = (
    df.drop(columns='CHAS')
    .assign(RAD=df['RAD'].cat.codes)
    .corr(method='spearman', numeric_only=True)
)
heatmap_correlacao(corr_spearman, 'Matriz de Correlação de Spearman', 'heatmap_spearman')

`RAD` é categórica. Por isso, a análise avalia a associação de `RAD` com `TAX` pela distribuição de `TAX` em cada nível de `RAD`, e não pela correlação de Pearson. Para os preditores numéricos e `CHAS`, o cálculo do VIF é o normal. `RAD` entra como 8 indicadoras, e para ela a análise usa o **VIF generalizado (GVIF)**, que mede a colinearidade do bloco inteiro de indicadoras. Para comparar variáveis com graus de liberdade (GL) diferentes, a análise usa `GVIF^(1/(2·GL))`. Os valores de referência são **2,24** (equivalente a VIF 5) e **3,16** (equivalente a VIF 10).

In [29]:
df.groupby('RAD', observed=True)['TAX'].agg(['count', 'min', 'median', 'max'])

,count,min,median,max
RAD,,,,
1,20,198,284.5,422
2,24,188,270.0,348
3,43,193,233.0,469
4,110,224,306.0,711
5,115,187,358.0,403
6,26,293,391.0,432
7,17,222,330.0,330
8,24,284,307.0,307
24,132,666,666.0,666


In [30]:
colunas_explicativas = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
    'PTRATIO', 'B', 'LSTAT']

calcular_vif(df, colunas_explicativas)

,Variável,GL,GVIF,GVIF^(1/(2·GL))
0,TAX,1,9.903660,3.147008
1,NOX,1,4.666367,2.160177
2,INDUS,1,4.401600,2.097999
3,DIS,1,4.177949,2.044003
4,AGE,1,2.910909,1.706139
5,ZN,1,2.492544,1.578779
6,LSTAT,1,2.282035,1.510641
7,PTRATIO,1,2.187746,1.479103
8,CRIM,1,1.797892,1.340855
9,RM,1,1.750337,1.323003


In [31]:
colunas_explicativas = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'B', 'LSTAT']

calcular_vif(df, colunas_explicativas)

,Variável,GL,GVIF,GVIF^(1/(2·GL))
0,NOX,1,4.656489,2.157890
1,DIS,1,4.169139,2.041847
2,INDUS,1,3.599460,1.897224
3,AGE,1,2.904770,1.704339
4,ZN,1,2.414857,1.553981
5,LSTAT,1,2.281637,1.510509
6,PTRATIO,1,2.178208,1.475875
7,CRIM,1,1.797564,1.340733
8,RM,1,1.750173,1.322941
9,B,1,1.343795,1.159222


In [32]:
colunas_explicativas2 = ['logCRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'logLSTAT']

print(calcular_vif(df, colunas_explicativas2))

    Variável  GL       GVIF  GVIF^(1/(2·GL))
0    logCRIM   1   8.309379         2.882599
1        NOX   1   4.749737         2.179389
2        DIS   1   4.142365         2.035280
3      INDUS   1   3.618410         1.902212
4        AGE   1   3.122714         1.767120
5   logLSTAT   1   3.058498         1.748856
6      logZN   1   2.825872         1.681033
7    PTRATIO   1   2.402946         1.550144
8         RM   1   2.023197         1.422391
9        RAD   8  16.303419         1.190604
10      CHAS   1   1.093650         1.045777


In [33]:
colunas_explicativas2 = ['CRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'logLSTAT']

print(calcular_vif(df, colunas_explicativas2))

    Variável  GL      GVIF  GVIF^(1/(2·GL))
0        NOX   1  4.643204         2.154809
1        DIS   1  4.156783         2.038819
2      INDUS   1  3.624497         1.903811
3        AGE   1  3.102527         1.761399
4   logLSTAT   1  3.032868         1.741513
5      logZN   1  2.835089         1.683772
6    PTRATIO   1  2.340675         1.529926
7         RM   1  2.018737         1.420823
8       CRIM   1  1.744971         1.320974
9        RAD   8  6.296693         1.121875
10      CHAS   1  1.088447         1.043286


Número de condição da matriz de design (preditores padronizados + intercepto). Valores acima de ~30 indicam multicolinearidade relevante:

In [34]:
especificacoes = {
    "Original (com TAX)": ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'],
    "Original (sem TAX)": ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'B', 'LSTAT'],
    "Transformada (logCRIM)": ['logCRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'logLSTAT'],
    "Transformada (CRIM)": ['CRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'logLSTAT'],
}

pd.Series(
    {nome: calcular_numero_condicao(df, cols) for nome, cols in especificacoes.items()},
    name="Número de condição"
).round(2)

Original (com TAX)        13.09
Original (sem TAX)        11.24
Transformada (logCRIM)    12.34
Transformada (CRIM)       11.13
Name: Número de condição, dtype: float64

### Conclusão - Multicolinearidade

O cálculo do VIF usa **só as variáveis explicativas**. A variável resposta `MEDV` não entra nesse cálculo, porque o VIF mede quanto os outros preditores explicam cada preditor. `RAD` é categórica, e por isso a análise usa o **VIF generalizado (GVIF)**. O GVIF é igual ao VIF nas variáveis com um grau de liberdade. Para comparar variáveis com graus de liberdade diferentes, a análise usa `GVIF^(1/(2·GL))`, com valores de referência de 2,24 (VIF 5) e 3,16 (VIF 10).

A análise encontrou uma associação relevante entre `TAX` e `RAD`. Esse é o principal ponto de atenção para os modelos de regressão. No modelo com as duas variáveis, `TAX` tem GVIF de cerca de **9,90** (`GVIF^(1/(2·GL))` = 3,15, quase no limite de 3,16). O bloco de `RAD` tem GVIF de **18,88** com 8 graus de liberdade (`GVIF^(1/(2·GL))` = 1,20).

A tabela de `TAX` por nível de `RAD` explica esse resultado: **todas as 132 observações com `RAD = 24` têm `TAX = 666`**. Nos outros níveis, `TAX` varia entre 187 e 711. Assim, a indicadora de `RAD = 24` determina grande parte de `TAX`. A correlação de Spearman entre as duas variáveis, com `RAD` ordinal, é de **0,70**.

Para estudar esse comportamento, a análise calculou o GVIF de novo, sem `TAX`. Nesse caso, o GVIF de `RAD` cai para cerca de **6,35** (`GVIF^(1/(2·GL))` = 1,12), e os VIFs das outras variáveis ficam abaixo de 4,7. Essa redução mostra que grande parte da multicolinearidade do modelo inicial está associada à presença de `TAX` e `RAD` juntas.

Nas especificações com variáveis transformadas, há um segundo ponto de atenção: **`logCRIM` tem VIF de 8,31** (`GVIF^(1/(2·GL))` = 2,88, acima da referência de 2,24). Nesse caso, o GVIF de `RAD` sobe para 16,30. Na mesma especificação com `CRIM` na escala original, o VIF de `CRIM` é só 1,74 (com `RAD` em 6,30). A causa é que a transformação logarítmica deixa `CRIM` mais linearmente associada aos outros preditores, principalmente ao bloco de indicadoras de `RAD`. As Partes 2 e 3 indicaram `logCRIM` como uma transformação promissora, e por isso a escolha da forma funcional precisa considerar esse aumento de colinearidade.

O **número de condição** da matriz de design (preditores padronizados + intercepto, com `RAD` em indicadoras) é de 13,09 com `TAX` e de 11,24 sem `TAX`. É de 12,34 na especificação com `logCRIM` e de 11,13 na especificação com `CRIM`. Esses valores são maiores que os valores com `RAD` numérica, porque as 8 indicadoras acrescentam colunas e alguns níveis têm poucas observações. Mas todos ficam abaixo da referência de 30. Assim, não há multicolinearidade severa no conjunto inteiro. Os problemas encontrados são locais (`TAX`/`RAD` e `logCRIM`/`RAD`).

A variável `CHAS`, no cálculo como indicadora (0/1), tem VIF entre **1,09 e 1,10** em todas as especificações. Assim, `CHAS` não tem redundância relevante com os outros preditores.

A correlação de Spearman também apoia as conclusões da Parte 3. A associação entre `LSTAT` e `MEDV` é de **−0,82** por Spearman, contra **−0,56** por Pearson. Esse resultado mostra uma relação monotônica forte, mas não linear, compatível com a transformação `logLSTAT`.

Com esses resultados, **a remoção de `TAX` é uma alternativa**, com `RAD` no modelo como fator categórico. Mas a decisão não depende só dos valores de GVIF. Também é necessário considerar a relevância teórica de cada variável e comparar o ajuste e os diagnósticos dos modelos com especificações diferentes. Uma alternativa é ajustar modelos com `TAX` e sem `TAX`, com as outras condições iguais, e comparar os resultados. Se houver motivo, a análise também pode estudar a remoção de `RAD` ou a sua redução a uma indicadora de `RAD = 24`.

Em resumo, **a multicolinearidade entre `TAX` e `RAD` é o principal problema desta etapa**. O segundo problema é o aumento de colinearidade com a transformação `logCRIM`. Os outros preditores têm VIF relativamente baixo. Esses resultados usam as 5 observações suspeitas (ver Parte 1). Se essas observações saírem, é necessário rever esses resultados.

## Adicionando informações ao conjunto de dados

> O arquivo processado tem todas as 511 observações, com as 5 linhas suspeitas (índices 506 a 510) e as 5 observações com `RM` ausente. O tratamento dessas observações precisa acontecer antes do ajuste do modelo, no notebook [`02-MRLM.ipynb`](02-MRLM.ipynb).

In [35]:
df.to_csv('../data/housing_processed.csv', index=False)